### 集成测试
如果你想要对保存的模型进行集成测试而不是通过每次都全部运行的方式进行 <br>
这个文件你可以使用 <br>
注 ：文件中大部分避开了传参数 除非必须传参数 大部分采取全局变量定义

把所有的全局变量和需要导入的包全部放在这里 如果需要不读数据打分 可以只运行此code

In [1]:
from tqdm import tqdm
import os
import re

pattern = r".*val_wei-(?P<wei_value>-?\d+\.\d+).*\.pth$"
sign = 'base'
root_path = '/home/user96/Mixer_v4'
train_save_name = 'model_train'
train_save_path = f'{root_path}/{train_save_name}'
month = 24
season_list = ["2023q1","2023q2","2023q3","2023q4","2024q1","2024q2","2024q3","2024q4"]
kfold = 4
period_list = [i+1 for i in range(kfold)]

from argparse import ArgumentParser
from model import *

parser = ArgumentParser(allow_abbrev=False)

parser.add_argument('--factor_num' , type= int ,default= 1400)

args , unknown_args = parser.parse_known_args()


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = PredictModel(args=args)

model.to(device=device)

PredictModel(
  (input_layter): Sequential(
    (0): Linear(in_features=1400, out_features=256, bias=True)
    (1): LeakyReLU(negative_slope=0.01, inplace=True)
    (2): Dropout(p=0.2, inplace=False)
  )
  (net): Sequential(
    (0): Linear(in_features=256, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): GELU(approximate='none')
    (3): Dropout(p=0.1, inplace=False)
  )
  (mixer): MultTime2dMixer(
    (mix_layer): Mixer2dTriU(
      (LN_1): LayerNorm((1, 128), eps=1e-05, elementwise_affine=True)
      (LN_2): LayerNorm((1, 128), eps=1e-05, elementwise_affine=True)
      (timeMixer): TriU(
        (triU): ParameterList(
            (0): Object of type: Linear
          (0): Linear(in_features=1, out_features=1, bias=True)
        )
      )
      (channelMixer): MixerBlock(
        (dense_1): Linear(in_features=128, out_features=128, bias=True)
        (LN): GELU(approximate='none')
        (dense_2): Linear

#### 下面对model的名称进行读取，注意我们只取每次训练的所有epoch的pth文件 并且根据名称中的val wei需要排序

定义一个pattern和根据pattern取浮点数对文件夹内文件名进行排序的函数 我们用val wei进行pattern的匹配可以直接进行排序

In [2]:
def filter_and_sort_files(directory, pattern, sort_key, reverse=False):
    """
    根据文件名中的特定数值筛选并排序文件
    
    :param directory: 要搜索的目录
    :param pattern: 文件名模式 正则表达式 
    :param sort_key: 用于排序的键名（在正则表达式中捕获的组名）
    :param reverse: 是否反向排序 默认False 即升序
    :return: 排序后的文件路径列表
    """
    # 编译正则表达式
    regex = re.compile(pattern)
    
    # 存储匹配结果
    matched_files = []
    
    # 遍历目录中的所有文件
    for filename in os.listdir(directory):
        filepath = os.path.join(directory, filename)
        if os.path.isfile(filepath):
            # 尝试匹配文件名
            match = regex.match(filename)
            if match:
                # 提取数值
                try:
                    value = float(match.group(sort_key))
                    matched_files.append((value, filepath))
                except (ValueError, KeyError):
                    # 如果无法转换为浮点数或键名不存在，跳过
                    continue
    
    # 按数值排序
    matched_files.sort(key=lambda x: x[0], reverse=reverse)
    
    # 返回文件路径和取得的val wei 方便传入ensembler
    return matched_files


如果你需要对集成模型进行打分 我们对所有的需要用到的数据和数据集构造类进行读取和导入

In [3]:
#我们只读一次数据即可 然后丢给data里面的dataset处理 用一样的方式得到某个季度的测试集
# from data import *

# 因子
fac_path = rf'/project/model_share/share_1_new/factor_data/fac20250613_out'
fac_name = rf'fac20250613'
# 标签
label_path = rf'/home/user96/label_data1/'
label_name = rf'label1'
# 流动性数据
liquid_path = rf'/home/user96/label_data1/'
liquid_name = rf'can_trade_amt1'

label_file_name = rf'{label_path}/{label_name}.fea'
liquid_file_name = rf'{label_path}/{liquid_name}.fea'
fac_file_name = f'{fac_path}/{fac_name}/{fac_name}.fea'

all_data = pd.read_feather(fac_file_name)
liquid_data = pd.read_feather(liquid_file_name).set_index("index")
ret_data = pd.read_feather(label_file_name).set_index("index")

date_list = list(all_data["date"].unique())
date_list = [x for x in date_list if x in ret_data.index and x in liquid_data.index]
date_list.sort()

all_data = all_data.set_index("date").sort_index()

all_data = all_data.loc[:, all_data.replace(0, np.nan).dropna(how="all", axis=1).columns]

factor_num = 1400

下面是模改一下data处理里的因子提取函数 只需要加入season的参数即可对每个季度进行读取 应该都是load 因为你运行的时候保证了因子已经筛选保存好了

In [4]:
def load_filter_factor(season):
    print('='*50)
    print(f"The factor for month{month} in {season} is already filtered before!")
    print("We use the existed factor directly!")
    all_name=None
    #you can define the method to read factor name 
    with open(rf'{train_save_path}/{season}-{month}-feature_map.fea','r') as file:
        all_name=file.read()
    
    factor_list=all_name.split('\n')
    #The last row is '\n' so I drop the ''
    factor_list=[x for x in factor_list if x!='']
    #we should keep the Code column
    filt_map=['Code']+factor_list
    
    return filt_map

def save_filter_factor(season):
    # add a judgement for existed file ,So we can use it to avoid doing the repeated thing
    file_path = rf'{train_save_path}/{season}-{month}-feature_map.fea'
    if os.path.exists(file_path):
        return load_filter_factor(season)
    # get the fliter factors , you can see the filter strategy in fac_filter.py
    factor_list = get_filter_factors(date_list,
                                        all_data,
                                        liquid_data,
                                        ret_data,
                                        season,
                                        month,
                                        factor_num)
    
    with open(file_path, 'w') as file:
        for factor_name in factor_list:
            file.write(rf'{factor_name}')
            file.write('\n')
            
    #we should keep the Code column
    filt_map=['Code']+factor_list

    return filt_map

定义一个函数 输入一个ensemble model <br>
让model像model里面的逻辑一样进行打分 这样打完分在test里面测试就行 <br>
注：此处的命名逻辑与model.py中的trainer需要保持一致 方便定位和寻找

In [5]:
def move_to_device(batch):
    inputs ,rets, times, code_values, liquids = batch
    inputs = [x.to(device) for x in inputs]
    rets = [x.to(device) for x in rets]
    liquids =  [x.to(device) for x in liquids]
    return inputs , rets , times , code_values , liquids
    
def mark_ensemble_model(model , season ,period, ensemble_method, ensemble_num):
    #这里我只需要test date list 除了season和datelist 其他参数不重要 随便填即可
    _ ,_ , test_date_list = get_train_date_split(  4 , date_list , season , 1 , 10 , 24)
    
    #读取因子
    factor_list = save_filter_factor(season)
    tmp_data = all_data[factor_list]
    test_data = DLDataset(tmp_data,test_date_list,factor_list,liquid_data,ret_data, stage='test')
        
    test_loader = DataLoader(test_data, batch_size=1, collate_fn=collate_fn,
                    num_workers=0, persistent_workers=False,
                    pin_memory=False, drop_last=False)

    model.eval()
    
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Test", leave=True):
            inputs ,rets, times, code_values, liquids = move_to_device(batch)
            for i in range(len(inputs)):
                input, ret ,time ,code ,liquid = inputs[i], rets[i] ,times[i] ,code_values[i] , liquids[i]
                #forward
                preds = model(input)
                res = pd.DataFrame(preds.detach().cpu().numpy(), index=code, columns=['value'])
                res.index.name = 'Code'
                test_ensemble_save_path = f'{root_path}/model_test/{sign}/ensemble-{ensemble_method}-top{ensemble_num}-m{month}-period{period}'#保持一致
                os.makedirs(test_ensemble_save_path,exist_ok=True)
                res.to_pickle(f'{test_ensemble_save_path}/{time}.pkl')
    return 

In [ ]:
import copy
from utils import *
import torch
ensemble_num = 5
ensemble_method = 'weighted_average'

ensembler = ModelEnsembler(ensemble_method,args,device)
season_list = ["2023q1","2023q2","2023q3","2023q4","2024q1","2024q2","2024q3","2024q4"]
def season_ensemble(season):
    for period in period_list:
        folder_name = f'{train_save_path}/{sign}/m{month}-{season}-period{period}'
        print(folder_name)
        assert(os.path.exists(folder_name))
        sort_file_name = filter_and_sort_files(folder_name,pattern,'wei_value',True)
        print(len(sort_file_name))
        
        #把指定数目的打分靠前的model加载进ensembler
        for i in range(ensemble_num):
            val_wei , model_path = sort_file_name[i]
            model.load_state_dict(torch.load(model_path))
            #防止浅拷贝
            model_copy = copy.deepcopy(model)
            ensembler.add_model(model=model_copy,val_metric=val_wei)

        #通过ensembler进行集成 并且保存到train的同一位置 调用函数进行打分
        ensemble_save_name = f'{folder_name}/ensemble-{ensemble_method}-top{ensemble_num}.pth'
        ensemble_model = ensembler.save_ensemble_model(ensemble_save_name)
        
        #每次使用完集成之后保存好了之后要清空集成器的model
        ensembler.clear()
        
        mark_ensemble_model(ensemble_model,season,period,ensemble_method,ensemble_num)   
for season in season_list:
    season_ensemble(season)

/home/user96/Mixer_v4/model_train/base/m24-2023q1-period1
0


IndexError: list index out of range

### 比较模型参数差异


由于集成参数无法提升ret 为了调查average是否对参数影响 决定打印各个model之间参数的欧氏距离和余弦相似度

In [ ]:
import torch

def compare_models(model1, model2, device):
    model1.eval()
    model2.eval()
    
    # 获取模型状态字典
    state_dict1 = model1.state_dict()
    state_dict2 = model2.state_dict()
    
    # 结果容器
    results = {
        "floating_diffs": {
            "cosine": 0.0,
            "param_details": []
        },
        "non_floating_diffs": {
            "num_different": 0,
            "param_details": {}
        }
    }
    
    # 浮点参数计数器
    num_floating_params = 0
    
    # 遍历所有参数
    for name, param1 in state_dict1.items():
        if name not in state_dict2:
            print(f"警告: {name} 在第二个模型中不存在")
            continue
            
        param2 = state_dict2[name]
        
        # 检查形状是否相同
        if param1.shape != param2.shape:
            print(f"警告: {name} 形状不同: {param1.shape} vs {param2.shape}")
            continue
        
        # 转换为相同设备
        param1 = param1.to(device)
        param2 = param2.to(device)
        
        # 处理浮点类型参数
        if param1.is_floating_point():
            # 展平参数以便计算
            flat1 = param1.flatten()
            flat2 = param2.flatten()
            
            # 计算余弦相似度
            dot_product = torch.dot(flat1, flat2).item()
            norm1 = torch.norm(flat1, p=2).item()
            norm2 = torch.norm(flat2, p=2).item()
            
            # 避免除以零
            if norm1 > 0 and norm2 > 0:
                cosine = dot_product / (norm1 * norm2)
            else:
                cosine = 1.0 if norm1 == norm2 else -1.0  # 处理零向量情况
            
            # 更新总体差异
            results["floating_diffs"]["cosine"] += cosine
            num_floating_params += 1
            
            # 保存参数级细节
            results["floating_diffs"]["param_details"].append({
                "name": name,
                "cosine": cosine,
                "shape": tuple(param1.shape)
            })
        else:
            # 处理非浮点类型参数
            if not torch.equal(param1, param2):
                results["non_floating_diffs"]["num_different"] += 1
                results["non_floating_diffs"]["param_details"][name] = (
                    param1.detach().clone(), 
                    param2.detach().clone()
                )
    
    # 计算平均余弦相似度和平均欧式距离
    if num_floating_params > 0:
        results["floating_diffs"]["cosine"] /= num_floating_params
    
    return results

def visualize_comparison(results, top_k):
    """可视化模型比较结果"""
    # 1. 浮点参数差异总结
    floating = results["floating_diffs"]
    print("\n" + "="*50)
    print("浮点参数差异总结")
    print("="*50)
    print(f"平均余弦相似度: {floating['cosine']:.4f}")
    
    # 显示差异最大的参数
    if floating["param_details"]:
        # 按余弦相似度排序
        sorted_by_cosine = sorted(
            floating["param_details"], 
            key=lambda x: x["cosine"]
        )[:top_k]
        
        print("\n余弦相似度最小的参数:")
        for param in sorted_by_cosine:
            print(f"  {param['name']} (形状{param['shape']}): {param['cosine']:.4f}")
    
    # 2. 非浮点参数差异
    non_floating = results["non_floating_diffs"]
    print("\n" + "="*50)
    print("非浮点参数差异")
    print("="*50)
    print(f"不同参数数量: {non_floating['num_different']}")
    
    if non_floating["param_details"]:
        print("\n不同的参数:")
        for name, (val1, val2) in non_floating["param_details"].items():
            print(f"  {name}:")
            print(f"    模型1: {val1.cpu().numpy()}")
            print(f"    模型2: {val2.cpu().numpy()}")
    
    print("="*50 + "\n")

可以自定义对比不同季度不同period内topk的相似度 保存到文件中方便阅读分析

In [ ]:
#这个函数的含义是 给定season和period 我们比较这其中 建议把上面的全局变量运行再进行测试
#topk代表我测试其中前topk之间两两相似度的均值
def test_epochs_model(season,period,file,topk):
    test_folder_name = f'{train_save_path}/{sign}/m{month}-{season}-period{period}'
    file.write('='*20)
    file.write(f'Now we test the {season} with period{period} top{topk} models difference!')
    file.write('='*20)
    file.write('\n')
    assert(os.path.exists(test_folder_name))
    sort_file_name = filter_and_sort_files(test_folder_name,pattern,'wei_value',True)
    all_cosines = []
    for i in range(topk-1):
        for j in range(i+1,topk):
            model1 = Stock_MLP(args=args)
            model1.to(device=device)
            model2 = Stock_MLP(args=args)
            model2.to(device=device)

            model1.load_state_dict(torch.load(sort_file_name[i][1]))
            model2.load_state_dict(torch.load(sort_file_name[j][1]))

            # 比较模型
            results = compare_models(model1, model2 ,device)

            # 可视化结果
            # visualize_comparison(results,5)

            # 高级分析：计算差异分布
            if results["floating_diffs"]["param_details"]:
                cosines = [p["cosine"] for p in results["floating_diffs"]["param_details"]]
                
                file.write(f"Analyse the difference between top{i+1} and top{j+1}:")
                file.write(f"余弦相似度 - 均值: {np.mean(cosines):.4f}, 最大值: {np.max(cosines):.4f}, 最小值: {np.min(cosines):.4f}\n")
            
                all_cosines.append(np.mean(cosines))
        
    file.write('='*20)
    file.write(f'Finally for {season} with period{period} top{topk} average cosine similarity is:{np.mean(all_cosines):.4f}')
    file.write('='*20)
    file.write('\n')
    

In [ ]:
for topk in [3,5,7,9]:
    difference_save_path = f'{root_path}/differs'
    os.makedirs(difference_save_path,exist_ok=True)
    file_name = f'{difference_save_path}/all_differ_for_top{topk}'
    with open(file_name,'w') as file:
        for season in season_list:
            for period in period_list:    
                test_epochs_model(season,period,file,topk)

下面为了引入新的集成思路 我们对同一季度 不同period之间最好的参数进行对比

In [ ]:
def test_periods_model(season,file):
    file.write('='*20)
    file.write(f'Now we test the {season} periods models difference!')
    file.write('='*20)
    file.write('\n')
    period_num = len(period_list)
    valwei_models = []
    for period in period_list:
        test_folder_name = f'{train_save_path}/{sign}/m{month}-{season}-period{period}'
        assert(os.path.exists(test_folder_name))
        sort_file_name = filter_and_sort_files(test_folder_name,pattern,'wei_value',True)
        valwei_models.append(sort_file_name[0])#都取最好的进行对比即可 保留val wei
    all_cosines = []
    for i in range(period_num-1):
        for j in range(i+1,period_num):
            model1 = Stock_MLP(args=args)
            model1.to(device=device)
            model2 = Stock_MLP(args=args)
            model2.to(device=device)

            model1.load_state_dict(torch.load(valwei_models[i][1]))
            model2.load_state_dict(torch.load(valwei_models[j][1]))

            # 比较模型
            results = compare_models(model1, model2 ,device)

            # 可视化结果
            # visualize_comparison(results,5)

            # 高级分析：计算差异分布
            if results["floating_diffs"]["param_details"]:
                cosines = [p["cosine"] for p in results["floating_diffs"]["param_details"]]
                
                file.write(f"Analyse the difference between period{i+1} and period{j+1}:")
                file.write(f"余弦相似度 - 均值: {np.mean(cosines):.4f}, 最大值: {np.max(cosines):.4f}, 最小值: {np.min(cosines):.4f}\n")
            
                all_cosines.append(np.mean(cosines))
        
    file.write('='*20)
    file.write(f'Finally for {season} different period average cosine similarity is:{np.mean(all_cosines):.4f}')
    file.write('='*20)
    file.write('\n')
    

In [ ]:
difference_save_path = f'{root_path}/differs'
os.makedirs(difference_save_path,exist_ok=True)
file_name = f'{difference_save_path}/all_differ_for_period'
with open(file_name,'w') as file:
    for season in season_list: 
        test_periods_model(season,file)

### 分析
经过上述的输出结果进行对比发现不同period之间的参数差别还是很大的 这样的集成可能对模型性能造成巨大的影响 <br>
当然也有可能0.7的相似度多period之后也可以有正向的影响 下面尝试一下各个季度不同period之间最好的model进行集成打分分析<br>
或者我们应该取的不是表现最好的模型之间进行集成 而应该选择相似度最大的模型？ 那如何定义多period之间的一组相似度最大呢？


In [ ]:
def mark_ensemble_period_model(model , season , ensemble_method):
    #这里我只需要test date list 除了season和datelist 其他参数不重要 随便填即可
    _ ,_ , test_date_list = get_train_date_split(  4 , date_list , season , 1 , 10 , 24)
    
    #读取因子
    factor_list = save_filter_factor(season)
    tmp_data = all_data[factor_list]
    test_data = DLDataset(tmp_data,test_date_list,factor_list,liquid_data,ret_data, stage='test')
        
    test_loader = DataLoader(test_data, batch_size=1, collate_fn=collate_fn,
                    num_workers=0, persistent_workers=False,
                    pin_memory=False, drop_last=False)

    model.eval()
    
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Test", leave=True):
            inputs ,rets, times, code_values, liquids = move_to_device(batch)
            for i in range(len(inputs)):
                input, ret ,time ,code ,liquid = inputs[i], rets[i] ,times[i] ,code_values[i] , liquids[i]
                #forward
                preds = model(input)
                res = pd.DataFrame(preds.detach().cpu().numpy(), index=code, columns=['value'])
                res.index.name = 'Code'
                test_ensemble_save_path = f'{root_path}/model_test/{sign}/ensemble-{ensemble_method}-allperiodbest'#保持一致
                os.makedirs(test_ensemble_save_path,exist_ok=True)
                res.to_pickle(f'{test_ensemble_save_path}/{time}.pkl')
    return 

In [ ]:
def periods_ensemble(season ,ensemble_method):
    period_num = len(period_list)
    valwei_models = []
    for period in period_list:
        folder_name = f'{train_save_path}/{sign}/m{month}-{season}-period{period}'
        assert(os.path.exists(folder_name))
        sort_file_name = filter_and_sort_files(folder_name,pattern,'wei_value',True)
        
        valwei_models.append(sort_file_name[0])
    
        
    #把指定数目的打分靠前的model加载进ensembler
    for i in range(period_num):
        val_wei , model_path = valwei_models[i]
        model1 = Stock_MLP(args=args)
        model1.to(device=device)
        model1.load_state_dict(torch.load(model_path))
        #防止浅拷贝
        model_copy = copy.deepcopy(model)
        ensembler.add_model(model=model_copy,val_metric=val_wei)

    #通过ensembler进行集成 并且保存到train的同一位置 调用函数进行打分
    ensemble_save_name = f'{root_path}/differs/ensemble-{season}-{ensemble_method}-allperiodbest.pth'
    ensemble_model = ensembler.save_ensemble_model(ensemble_save_name)
    
    #每次使用完集成之后保存好了之后要清空集成器的model
    ensembler.clear()
    
    mark_ensemble_period_model(ensemble_model,season,ensemble_method)  

In [ ]:
for season in season_list:
    periods_ensemble(season ,'average')

# 总结
毫无可能！！！！集成必须集成相似度不大的 否则灭顶之灾<br>

Now we test the ic and label ret from 20230101 to 20241231 The model Identify sign is base
The average IC is 0.031673 , while the average label ret is 0.075489